In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Bronze_Layer_Full_Audit") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.1.0,org.apache.hadoop:hadoop-aws:3.3.2") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

## Đọc dữ liệu và thống kê tổng quan

In [3]:
# Đọc toàn bộ dữ liệu Bronze
df_bronze = spark.read.format("delta").load("s3a://bronze/electronics_events")

# 1. Tổng số dòng và số cột
print(f"Tổng số bản ghi: {df_bronze.count():,}")
print(f"Số lượng cột: {len(df_bronze.columns)}")
print("-" * 30)

# 2. Schema chi tiết
df_bronze.printSchema()

Tổng số bản ghi: 885,129
Số lượng cột: 11
------------------------------
root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: long (nullable = true)
 |-- user_session: string (nullable = true)
 |-- ingestion_at: timestamp (nullable = true)
 |-- execution_date: string (nullable = true)



## Kiểm tra dữ liệu thiếu 

In [4]:
# Tính toán số lượng NULL và tỷ lệ % NULL cho từng cột
total_count = df_bronze.count()
null_df = df_bronze.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_bronze.columns
])

print("SỐ LƯỢNG DÒNG NULL TRONG TỪNG CỘT:")
null_df.show()

print("TỶ LỆ % NULL TRONG TỪNG CỘT:")
null_df.select([
    F.round((F.col(c) / total_count * 100), 2).alias(f"{c}_%_null") for c in df_bronze.columns
]).show()

SỐ LƯỢNG DÒNG NULL TRONG TỪNG CỘT:
+----------+----------+----------+-----------+-------------+------+-----+-------+------------+------------+--------------+
|event_time|event_type|product_id|category_id|category_code| brand|price|user_id|user_session|ingestion_at|execution_date|
+----------+----------+----------+-----------+-------------+------+-----+-------+------------+------------+--------------+
|         0|         0|         0|          0|       236219|212364|    0|      0|         165|           0|             0|
+----------+----------+----------+-----------+-------------+------+-----+-------+------------+------------+--------------+

TỶ LỆ % NULL TRONG TỪNG CỘT:
+-----------------+-----------------+-----------------+------------------+--------------------+------------+------------+--------------+-------------------+-------------------+---------------------+
|event_time_%_null|event_type_%_null|product_id_%_null|category_id_%_null|category_code_%_null|brand_%_null|price_%_null|

## Phân tích các chiều dữ liệu 

In [5]:
# Kiểm tra sự đa dạng của dữ liệu
stats = df_bronze.select(
    F.countDistinct("user_id").alias("Unique_Users"),
    F.countDistinct("product_id").alias("Unique_Products"),
    F.countDistinct("category_id").alias("Unique_Categories"),
    F.countDistinct("brand").alias("Unique_Brands")
)
print("THỐNG KÊ ĐỘ ĐA DẠNG:")
stats.show()

# Top 10 Thương hiệu xuất hiện nhiều nhất (bao gồm cả Null)
print("TOP 10 THƯƠNG HIỆU:")
df_bronze.groupBy("brand").count().orderBy(F.desc("count")).show(10)

THỐNG KÊ ĐỘ ĐA DẠNG:
+------------+---------------+-----------------+-------------+
|Unique_Users|Unique_Products|Unique_Categories|Unique_Brands|
+------------+---------------+-----------------+-------------+
|      407283|          53453|              718|          999|
+------------+---------------+-----------------+-------------+

TOP 10 THƯƠNG HIỆU:
+---------+------+
|    brand| count|
+---------+------+
|     null|212364|
|     asus| 27706|
| gigabyte| 27673|
|      msi| 24877|
|    palit| 24802|
|  samsung| 23208|
|      amd| 20110|
|    canon| 18438|
|panasonic| 11992|
|  pioneer| 11467|
+---------+------+
only showing top 10 rows



## Kiểm tra luồng sự kiện

In [6]:
print("PHÂN BỔ CÁC LOẠI HÀNH VI:")
event_dist = df_bronze.groupBy("event_type").count().orderBy(F.desc("count"))
event_dist.show()

# Tính tỷ lệ chuyển đổi đơn giản (Purchase / View)
view_count = df_bronze.filter(F.col("event_type") == "view").count()
purchase_count = df_bronze.filter(F.col("event_type") == "purchase").count()
print(f"Tỷ lệ chuyển đổi tổng thể (Purchase/View): {(purchase_count/view_count)*100:.2f}%")

PHÂN BỔ CÁC LOẠI HÀNH VI:
+----------+------+
|event_type| count|
+----------+------+
|      view|793748|
|      cart| 54035|
|  purchase| 37346|
+----------+------+

Tỷ lệ chuyển đổi tổng thể (Purchase/View): 4.71%


## Kiểm tra thời gian và Phân vùng

In [7]:
# Kiểm tra dữ liệu trải dài từ ngày nào đến ngày nào
time_range = df_bronze.select(
    F.min("event_time").alias("Start_Time"),
    F.max("event_time").alias("End_Time")
)
print("KHOẢNG THỜI GIAN CỦA DỮ LIỆU:")
time_range.show()

# Kiểm tra các phân vùng execution_date đã nạp vào MinIO
print("CÁC PHÂN VÙNG EXECUTION_DATE HIỆN CÓ:")
df_bronze.select("execution_date").distinct().orderBy("execution_date").show()

KHOẢNG THỜI GIAN CỦA DỮ LIỆU:
+-------------------+-------------------+
|         Start_Time|           End_Time|
+-------------------+-------------------+
|2020-09-24 11:57:06|2021-02-28 23:59:09|
+-------------------+-------------------+

CÁC PHÂN VÙNG EXECUTION_DATE HIỆN CÓ:
+--------------+
|execution_date|
+--------------+
|    2026-04-02|
+--------------+

